In [1]:
pip install google-genai

In [2]:
!pip install -q openai

In [3]:
# import os
# from google import genai
# from google.genai import types
# from openai import OpenAI

# # 1. Safely load your API Keys from Colab Secrets
# try:
#     from google.colab import userdata
#     os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_API_KEY')
#     os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
#     from google.colab import files as colab_files
#     IS_COLAB = True
# except Exception:
#     IS_COLAB = False
#     print("Running in a non-Colab environment. Browser downloads skipped.")

# # 2. Initialize Clients
# client_google = genai.Client()
# client_universal = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.environ.get("OPENROUTER_API_KEY")
# )

# def get_translation(text: str, model_id: str, target_language: str):
#     """Routes the translation request and returns (translated_text, token_dict)."""
#     prompt = (
#         f"Translate the following Punjabi text accurately into {target_language}. "
#         f"Maintain the precise meaning and context. Return only the {target_language} translation:\n\n"
#         f"{text}"
#     )

#     token_usage = {"total": 0, "in": 0, "out": 0}

#     try:
#         if "gemini" in model_id.lower() or "gemma" in model_id.lower():
#             response = client_google.models.generate_content(
#                 model=model_id,
#                 contents=prompt,
#                 config=types.GenerateContentConfig(temperature=0.2)
#             )

#             if hasattr(response, 'usage_metadata') and response.usage_metadata:
#                 token_usage["total"] = response.usage_metadata.total_token_count
#                 token_usage["in"] = response.usage_metadata.prompt_token_count
#                 token_usage["out"] = response.usage_metadata.candidates_token_count
#                 print(f"   📊 Tokens: {token_usage['total']} (In: {token_usage['in']} | Out: {token_usage['out']})")

#             if response.text:
#                 return response.text.strip(), token_usage
#             else:
#                 return f"API Error: Google returned an empty response.", token_usage

#         else:
#             response = client_universal.chat.completions.create(
#                 model=model_id,
#                 messages=[{"role": "user", "content": prompt}],
#                 temperature=0.2
#             )

#             if hasattr(response, 'usage') and response.usage:
#                 token_usage["total"] = response.usage.total_tokens
#                 token_usage["in"] = response.usage.prompt_tokens
#                 token_usage["out"] = response.usage.completion_tokens
#                 print(f"   📊 Tokens: {token_usage['total']} (In: {token_usage['in']} | Out: {token_usage['out']})")

#             content = response.choices[0].message.content
#             if content:
#                 return content.strip(), token_usage
#             else:
#                 return f"API Error: OpenRouter returned an empty response.", token_usage

#     except Exception as e:
#         return f"Error executing {model_id}: {str(e)}", token_usage

# def generate_judge_report(original_punjabi: str, translations: dict, target_language: str):
#     """Uses Llama 3.3 to evaluate translations and tracks its token usage."""
#     formatted_outputs = ""
#     for model, output in translations.items():
#         formatted_outputs += f"### Model Target: {model}\n{output}\n\n"

#     judge_prompt = f"""
#     You are an expert AI Benchmark Judge. Evaluate these Punjabi-to-{target_language} translations.
#     ---
#     ORIGINAL PUNJABI TRANSCRIPT:
#     {original_punjabi}
#     ---
#     CANDIDATE TRANSLATIONS:
#     {formatted_outputs}
#     ---
#     Generate a precise Evaluation Report in Markdown. You must include:
#     1. A comparison table scoring each model (1-10) on Accuracy, Fluency, and Cultural Nuance.
#     2. A brief analysis of any mistakes or strengths for each model.
#     3. A clear final verdict on which model is best suited for production workloads.
#     """

#     print(f"\n🦙 Analyzing translations and compiling the {target_language} judge evaluation scorecard...")
#     judge_tokens = {"total": 0, "in": 0, "out": 0}

#     try:
#         response = client_universal.chat.completions.create(
#             model="meta-llama/llama-3.3-70b-instruct",
#             messages=[{"role": "user", "content": judge_prompt}],
#             temperature=0.1
#         )

#         # Capture Judge Token Usage
#         if hasattr(response, 'usage') and response.usage:
#             judge_tokens["total"] = response.usage.total_tokens
#             judge_tokens["in"] = response.usage.prompt_tokens
#             judge_tokens["out"] = response.usage.completion_tokens
#             print(f"   📊 Judge Tokens: {judge_tokens['total']} (In: {judge_tokens['in']} | Out: {judge_tokens['out']})")

#         content = response.choices[0].message.content
#         if content:
#             return content.strip(), judge_tokens
#         else:
#             return f"API Error: OpenRouter returned an empty response.", judge_tokens
#     except Exception as e:
#         return f"Error executing Judge: {str(e)}", judge_tokens

# def process_and_download_translations(input_file_path: str, models_to_test: dict, target_language: str):
#     if not os.path.exists(input_file_path):
#         print(f"Error: The file '{input_file_path}' was not found. Please upload it first.")
#         return

#     base_name, _ = os.path.splitext(os.path.basename(input_file_path))
#     with open(input_file_path, "r", encoding="utf-8") as f:
#         punjabi_text = f.read().strip()

#     translations = {}
#     all_token_data = {}

#     print(f"\n🌍 STARTING {target_language.upper()} TRANSLATION PIPELINE...")
#     for short_name, model_id in models_to_test.items():
#         print(f"🚀 Translating with {model_id}...")

#         translated_text, tokens = get_translation(punjabi_text, model_id, target_language)

#         translations[short_name] = translated_text
#         all_token_data[short_name] = tokens

#         output_filename = f"{base_name}_{short_name}_{target_language.lower()}.txt"
#         with open(output_filename, "w", encoding="utf-8") as out_f:
#             out_f.write(translated_text)
#         print(f"💾 Saved Translation: {output_filename}")
#         if IS_COLAB:
#             colab_files.download(output_filename)

#     # --- BUILD & SAVE JUDGE REPORT ---
#     report_text, judge_tokens = generate_judge_report(punjabi_text, translations, target_language)
#     report_filename = f"{base_name}_{target_language.lower()}_evaluation_report.md"
#     with open(report_filename, "w", encoding="utf-8") as rep_f:
#         rep_f.write(report_text)

#     print(f"💾 Saved Benchmark Report: {report_filename}")
#     if IS_COLAB:
#         colab_files.download(report_filename)

#     # --- BUILD & SAVE GRAND TOTAL TOKEN REPORT ---
#     token_report_text = f"### 📊 {target_language} Translation Token Consumption Report\n\n"
#     token_report_text += "| Model | Task | Total Tokens | Prompt (In) | Completion (Out) |\n"
#     token_report_text += "|---|---|---|---|---|\n"

#     grand_total = 0
#     grand_in = 0
#     grand_out = 0

#     # Add Translation Models
#     for model, tokens in all_token_data.items():
#         token_report_text += f"| {model} | Translation | {tokens['total']} | {tokens['in']} | {tokens['out']} |\n"
#         grand_total += tokens['total']
#         grand_in += tokens['in']
#         grand_out += tokens['out']

#     # Add Judge Model
#     token_report_text += f"| Llama-3.3-Judge | Evaluation | {judge_tokens['total']} | {judge_tokens['in']} | {judge_tokens['out']} |\n"
#     grand_total += judge_tokens['total']
#     grand_in += judge_tokens['in']
#     grand_out += judge_tokens['out']

#     # Add Grand Total Row
#     token_report_text += f"| **GRAND TOTAL** | **ALL** | **{grand_total}** | **{grand_in}** | **{grand_out}** |\n"

#     token_filename = f"{base_name}_{target_language.lower()}_token_usage.md"
#     with open(token_filename, "w", encoding="utf-8") as tok_f:
#         tok_f.write(token_report_text)
#     print(f"\n💾 Saved Token Report: {token_filename}")
#     if IS_COLAB:
#         colab_files.download(token_filename)

#     print(f"\n✅ {target_language} Execution complete! All files downloaded successfully.")

In [4]:
# # --- EXECUTION ---
# my_ultimate_models = {
#     "gemini-2.5-flash": "gemini-2.5-flash",
#     "gemini-2.5-flash-lite": "gemini-2.5-flash-lite",
#     "gpt-4o-mini": "openai/gpt-4o-mini",
#     "deepseek-v4-flash": "deepseek/deepseek-v4-flash",
#     "qwen-3-8b": "Qwen/Qwen3-VL-8B-Instruct",
#     "llama-3.3-70b": "meta-llama/llama-3.3-70b-instruct"
# }

# uploaded_file_name = "Garden2006_turbo.txt"

# # This triggers the Hindi pipeline and calculates the grand total tokens
# process_and_download_translations(uploaded_file_name, my_ultimate_models, target_language="Hindi")


🌍 STARTING HINDI TRANSLATION PIPELINE...
🚀 Translating with gemini-2.5-flash...
💾 Saved Translation: Garden2006_turbo_gemini-2.5-flash_hindi.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 Translating with gemini-2.5-flash-lite...
💾 Saved Translation: Garden2006_turbo_gemini-2.5-flash-lite_hindi.txt


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>


💾 Saved Token Report: Garden2006_turbo_hindi_token_usage.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Hindi Execution complete! All files downloaded successfully.


In [9]:
# from google.colab import files as colab_files

# # 1. Define the exact token metrics from your full translation + evaluation run
# translation_token_metrics = [
#     {"model": "gemini-2.5-flash", "task": "Translation", "total": 0, "in": 0, "out": 0},
#     {"model": "gemini-2.5-flash-lite", "task": "Translation", "total": 0, "in": 0, "out": 0},
#     {"model": "gpt-4o-mini", "task": "Translation", "total": 4128, "in": 2572, "out": 1556},
#     {"model": "deepseek-v4-flash", "task": "Translation", "total": 7487, "in": 4252, "out": 3235},
#     {"model": "qwen-2.5-7b", "task": "Translation", "total": 11781, "in": 7217, "out": 4564},
#     {"model": "llama-3.3-70b", "task": "Translation", "total": 3724, "in": 2767, "out": 957},
#     {"model": "Llama-3.3-Judge", "task": "Evaluation", "total": 17631, "in": 17101, "out": 530}
# ]

# # 2. Structure the Markdown Content
# summary_filename = "Garden2006_turbo_hindi_token_usage.md"

# summary_markdown = f"""### 📊 Hindi Translation Token Consumption Report

# | Model | Task | Total Tokens | Prompt (In) | Completion (Out) |
# | :--- | :--- | :--- | :--- | :--- |
# """

# grand_total = 0
# grand_in = 0
# grand_out = 0

# for metric in translation_token_metrics:
#     summary_markdown += f"| {metric['model']} | {metric['task']} | {metric['total']} | {metric['in']} | {metric['out']} |\n"
#     grand_total += metric['total']
#     grand_in += metric['in']
#     grand_out += metric['out']

# # Add the Grand Total Row
# summary_markdown += f"| **GRAND TOTAL** | **ALL** | **{grand_total}** | **{grand_in}** | **{grand_out}** |\n"

# # 3. Write data to the file
# with open(summary_filename, "w", encoding="utf-8") as sum_f:
#     sum_f.write(summary_markdown)

# print(f"📁 Created separate summary archive: {summary_filename}")

# # 4. Trigger the download to your machine
# colab_files.download(summary_filename)

📁 Created separate summary archive: Garden2006_turbo_hindi_token_usage.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

punjabi ----> report

In [7]:
import os
from google import genai
from google.genai import types
from openai import OpenAI
from google.colab import userdata
from google.colab import files as colab_files

# 1. Safely load API Keys from Secrets
try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
    IS_COLAB = True
except Exception:
    IS_COLAB = False
    print("Running in a non-Colab environment. Browser downloads skipped.")

# 2. Initialize Clients
client_google = genai.Client()
client_universal = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY")
)

# 3. File Parameters
input_file_path = "Garden2006_turbo.txt"
base_name, _ = os.path.splitext(os.path.basename(input_file_path))

if not os.path.exists(input_file_path):
    raise FileNotFoundError(f"Please ensure '{input_file_path}' is uploaded to your Colab side panel.")

# Read Original Punjabi Transcript
with open(input_file_path, "r", encoding="utf-8") as f:
    punjabi_text = f.read().strip()

# 4. Construct Direct Analysis Prompt
direct_prompt = f"""
You are an expert linguistic analyst. Analyze the following original Punjabi transcript and generate a comprehensive structural and contextual report in Hindi.

ORIGINAL PUNJABI TRANSCRIPT:
{punjabi_text}

Your report must be written fully in Hindi and include:
1. Summary (सारांश): A detailed summary of the core themes and content.
2. Key Topics (मुख्य विषय): Break down the primary discussions or chapters.
3. Linguistic Nuances (भाषाई बारीकियां): Note any specific cultural idioms, tone, or complex terminology used in the Punjabi text and how they map contextually.

Format the output cleanly in Markdown.
"""

# The remaining models group to evaluate
remaining_models = {
    "gemini-2.5-flash": "gemini-2.5-flash",
    "gemini-2.5-flash-lite": "gemini-2.5-flash-lite",
    "gpt-4o-mini": "openai/gpt-4o-mini",
    "deepseek-v4-flash": "deepseek/deepseek-v4-flash",
    "llama-3.3-70b": "meta-llama/llama-3.3-70b-instruct",
    "qwen3-8b": "Qwen/Qwen3-VL-8B-Instruct"
}

print("🌍 STARTING DIRECT REPORT GENERATION WITH REMAINING MODELS...")
report_token_metrics = []

# 5. Process Execution Loop
for short_name, model_id in remaining_models.items():
    print(f"\n🚀 Generating Hindi report using {short_name}...")
    token_usage = {"total": 0, "in": 0, "out": 0}
    report_text = ""

    try:
        if "gemini" in model_id.lower():
            # Call Native Google SDK
            response = client_google.models.generate_content(
                model=model_id,
                contents=direct_prompt,
                config=types.GenerateContentConfig(temperature=0.3)
            )
            report_text = response.text.strip()

            # Correcting the Google metadata token tracking lookup format
            if hasattr(response, 'usage_metadata') and response.usage_metadata:
                token_usage["total"] = response.usage_metadata.total_token_count
                token_usage["in"] = response.usage_metadata.prompt_token_count
                token_usage["out"] = response.usage_metadata.candidates_token_count

        else:
            # Call OpenRouter API
            response = client_universal.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": direct_prompt}],
                temperature=0.3
            )
            report_text = response.choices[0].message.content.strip()

            if hasattr(response, 'usage') and response.usage:
                token_usage["total"] = response.usage.total_tokens
                token_usage["in"] = response.usage.prompt_tokens
                token_usage["out"] = response.usage.completion_tokens

        # Save and Download Report
        report_filename = f"{base_name}_{short_name}_direct_report.md"
        with open(report_filename, "w", encoding="utf-8") as rep_f:
            rep_f.write(report_text)

        print(f"💾 Saved: {report_filename}")
        if IS_COLAB:
            colab_files.download(report_filename)

        # Log results for final summary table
        report_token_metrics.append({
            "model": short_name,
            "total": token_usage["total"],
            "in": token_usage["in"],
            "out": token_usage["out"]
        })

    except Exception as e:
        print(f"❌ Error running {short_name}: {str(e)}")

# 6. Print Consolidated Token Usage Summary
print("\n" + "="*55)
print("📊 CONSOLIDATED TOKEN CONSUMPTION SUMMARY FOR DIRECT REPORT")
print("="*55)
print(f"{'Model Name':<22} | {'Total Tokens':<12} | {'Prompt (In)':<11} | {'Output (Out)':<12}")
print("-"*55)
for metric in report_token_metrics:
    print(f"{metric['model']:<22} | {metric['total']:<12,} | {metric['in']:<11,} | {metric['out']:<12,}")
print("="*55)

🌍 STARTING DIRECT REPORT GENERATION WITH REMAINING MODELS...

🚀 Generating Hindi report using gemini-2.5-flash...
💾 Saved: Garden2006_turbo_gemini-2.5-flash_direct_report.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🚀 Generating Hindi report using gemini-2.5-flash-lite...
❌ Error running gemini-2.5-flash-lite: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

🚀 Generating Hindi report using gpt-4o-mini...
💾 Saved: Garden2006_turbo_gpt-4o-mini_direct_report.md


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>


🚀 Generating Hindi report using qwen3-8b...
💾 Saved: Garden2006_turbo_qwen3-8b_direct_report.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📊 CONSOLIDATED TOKEN CONSUMPTION SUMMARY FOR DIRECT REPORT
Model Name             | Total Tokens | Prompt (In) | Output (Out)
-------------------------------------------------------
gemini-2.5-flash       | 8,074        | 2,767       | 2,713       
gpt-4o-mini            | 3,298        | 2,677       | 621         
deepseek-v4-flash      | 8,451        | 4,366       | 4,085       
llama-3.3-70b          | 8,740        | 7,828       | 912         
qwen3-8b               | 40,108       | 7,340       | 32,768      


In [8]:
from google.colab import files as colab_files

# 1. Define your exact direct report token metrics from your terminal run
report_token_metrics = [
    {"model": "gemini-2.5-flash", "status": "Success", "total": 8074, "in": 2767, "out": 2713},
    {"model": "gemini-2.5-flash-lite", "status": "Failed (503)", "total": 0, "in": 0, "out": 0},
    {"model": "gpt-4o-mini", "status": "Success", "total": 3298, "in": 2677, "out": 621},
    {"model": "deepseek-v4-flash", "status": "Success", "total": 8451, "in": 4366, "out": 4085},
    {"model": "llama-3.3-70b", "status": "Success", "total": 8740, "in": 7828, "out": 912},
    {"model": "qwen3-8b", "status": "Success", "total": 40108, "in": 7340, "out": 32768}
]

# 2. Structure the Markdown Content
summary_filename = "Garden2006_turbo_direct_report_summary.md"

summary_markdown = f"""# 📊 Direct Transcript-to-Report Token Summary
**Source File:** Garden2006_turbo.txt
**Task:** Direct Punjabi to Hindi Summary Report Generation

## 📈 Token Consumption Metrics Table

| Model Name | Status | Total Tokens | Prompt Tokens (In) | Output Tokens (Out) |
| :--- | :--- | :--- | :--- | :--- |
"""

for metric in report_token_metrics:
    summary_markdown += f"| {metric['model']} | {metric['status']} | {metric['total']:,} | {metric['in']:,} | {metric['out']:,} |\n"

# 3. Write data to the file
with open(summary_filename, "w", encoding="utf-8") as sum_f:
    sum_f.write(summary_markdown)

print(f"📁 Created separate summary archive: {summary_filename}")

# 4. Trigger the download to your machine
colab_files.download(summary_filename)

📁 Created separate summary archive: Garden2006_turbo_direct_report_summary.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

punjabi ---> hindi translate --> report


In [10]:
import os
from google import genai
from google.genai import types
from openai import OpenAI
from google.colab import userdata
from google.colab import files as colab_files

# 1. Load API Keys from Secrets Safely
try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
    IS_COLAB = True
except Exception:
    IS_COLAB = False
    print("Running in a non-Colab environment. Browser downloads skipped.")

# 2. Initialize Core Clients
client_google = genai.Client()
client_universal = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY")
)

# 3. File Configurations
input_file_path = "Garden2006_turbo.txt"
base_name, _ = os.path.splitext(os.path.basename(input_file_path))
target_language = "Hindi"

if not os.path.exists(input_file_path):
    raise FileNotFoundError(f"Please ensure '{input_file_path}' is uploaded to your Colab side panel.")

with open(input_file_path, "r", encoding="utf-8") as f:
    punjabi_text = f.read().strip()

# Defining the specific target model collection
models_to_test = {
    "gemini-2.5-flash": "gemini-2.5-flash",
    "gpt-4o-mini": "openai/gpt-4o-mini",
    "deepseek-v4-flash": "deepseek/deepseek-v4-flash",
    "llama-3.3-70b": "meta-llama/llama-3.3-70b-instruct",
    "qwen3-8b": "Qwen/Qwen3-VL-8B-Instruct",
    "gemini-2.5-flash-lite":"gemini-2.5-flash-lite"
}

print(f"🌍 INITIATING COMPLETE PIPELINE: PUNJABI ➔ {target_language.upper()} TRANSLATION ➔ REPORT GENERATION")
final_pipeline_metrics = []

# 4. Process Execution Loop
for short_name, model_id in models_to_test.items():
    print(f"\n──────────────────────────────────────────────────")
    print(f"🚀 Processing Model: {short_name}")
    print(f"──────────────────────────────────────────────────")

    trans_tokens = {"total": 0, "in": 0, "out": 0}
    report_tokens = {"total": 0, "in": 0, "out": 0}
    translated_text = ""
    report_text = ""

    # --- STEP A: TRANSLATION PHASE ---
    translation_prompt = (
        f"Translate the following Punjabi text accurately into {target_language}. "
        f"Maintain the precise meaning and context. Return only the {target_language} translation:\n\n"
        f"{punjabi_text}"
    )

    try:
        if "gemini" in model_id.lower():
            response_trans = client_google.models.generate_content(
                model=model_id,
                contents=translation_prompt,
                config=types.GenerateContentConfig(temperature=0.2)
            )
            translated_text = response_trans.text.strip()
            if hasattr(response_trans, 'usage_metadata') and response_trans.usage_metadata:
                trans_tokens["total"] = response_trans.usage_metadata.total_token_count
                trans_tokens["in"] = response_trans.usage_metadata.prompt_token_count
                trans_tokens["out"] = response_trans.usage_metadata.candidates_token_count
        else:
            response_trans = client_universal.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": translation_prompt}],
                temperature=0.2
            )
            translated_text = response_trans.choices[0].message.content.strip()
            if hasattr(response_trans, 'usage') and response_trans.usage:
                trans_tokens["total"] = response_trans.usage.total_tokens
                trans_tokens["in"] = response_trans.usage.prompt_tokens
                trans_tokens["out"] = response_trans.usage.completion_tokens

        # Save and download translation file
        trans_file = f"{base_name}_{short_name}_{target_language.lower()}.txt"
        with open(trans_file, "w", encoding="utf-8") as tf:
            tf.write(translated_text)
        print(f"   💾 Saved Translation File: {trans_file}")
        if IS_COLAB:
            colab_files.download(trans_file)

    except Exception as e:
        print(f"   ❌ Translation Phase Failed for {short_name}: {str(e)}")
        continue

    # --- STEP B: DIRECT REPORT GENERATION PHASE ---
    report_prompt = f"""
    You are an expert linguistic analyst. Analyze the following original Punjabi transcript and its corresponding {target_language} translation.
    Generate a comprehensive structural and contextual report in Hindi.

    ORIGINAL PUNJABI TRANSCRIPT:
    {punjabi_text}

    {target_language.upper()} TRANSLATION PRODUCED BY THIS MODEL:
    {translated_text}

    Your report must be written fully in Hindi and include:
    1. Summary (सारांश): A detailed summary of the core themes and content.
    2. Key Topics (मुख्य विषय): Break down the primary discussions or chapters.
    3. Linguistic Nuances (भाषाई बारीकियां): Note any specific cultural idioms, tone, or complex terminology and how they mapped contextually.

    Format the output cleanly in Markdown.
    """

    try:
        if "gemini" in model_id.lower():
            response_rep = client_google.models.generate_content(
                model=model_id,
                contents=report_prompt,
                config=types.GenerateContentConfig(temperature=0.3)
            )
            report_text = response_rep.text.strip()
            if hasattr(response_rep, 'usage_metadata') and response_rep.usage_metadata:
                report_tokens["total"] = response_rep.usage_metadata.total_token_count
                report_tokens["in"] = response_rep.usage_metadata.prompt_token_count
                report_tokens["out"] = response_rep.usage_metadata.candidates_token_count
        else:
            response_rep = client_universal.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": report_prompt}],
                temperature=0.3
            )
            report_text = response_rep.choices[0].message.content.strip()
            if hasattr(response_rep, 'usage') and response_rep.usage:
                report_tokens["total"] = response_rep.usage.total_tokens
                report_tokens["in"] = response_rep.usage.prompt_tokens
                report_tokens["out"] = response_rep.usage.completion_tokens

        # Save and download individual analysis report file
        report_file = f"{base_name}_{short_name}_hindi_report.md"
        with open(report_file, "w", encoding="utf-8") as rf:
            rf.write(report_text)
        print(f"   💾 Saved Analysis Report: {report_file}")
        if IS_COLAB:
            colab_files.download(report_file)

    except Exception as e:
        print(f"   ❌ Report Phase Failed for {short_name}: {str(e)}")
        continue

    # --- STEP C: CALCULATE GRAND TOTAL ---
    combined_total = trans_tokens["total"] + report_tokens["total"]
    combined_in = trans_tokens["in"] + report_tokens["in"]
    combined_out = trans_tokens["out"] + report_tokens["out"]

    final_pipeline_metrics.append({
        "model": short_name,
        "trans_total": trans_tokens["total"],
        "report_total": report_tokens["total"],
        "grand_in": combined_in,
        "grand_out": combined_out,
        "grand_total": combined_total
    })

# 5. Build and Save Consolidated Summary Log
summary_filename = f"{base_name}_pipeline_grand_total_summary.md"

summary_markdown = f"""# 📊 Full End-to-End Pipeline Token Summary
**Source Document:** {input_file_path}
**Workflow:** Punjabi Transcript ➔ Hindi Translation ➔ Independent Hindi Analysis Report

## 📈 Combined Token Footprint (Translation + Report Tasks)

| Model Name | Translation Tokens | Report Tokens | Grand Total Input (Prompt) | Grand Total Output (Completion) | Combined Grand Total |
| :--- | :--- | :--- | :--- | :--- | :--- |
"""

for row in final_pipeline_metrics:
    summary_markdown += (
        f"| {row['model']} | {row['trans_total']:,} | {row['report_total']:,} "
        f"| {row['grand_in']:,} | {row['grand_out']:,} | **{row['grand_total']:,}** |\n"
    )

with open(summary_filename, "w", encoding="utf-8") as sf:
    sf.write(summary_markdown)

print(f"\n📁 Created separate archive dataset summary: {summary_filename}")
if IS_COLAB:
    colab_files.download(summary_filename)

print("\n✅ All processes finished execution. All transcript texts and analytical markdown file structures have been compiled and sent to your downloads folder.")

🌍 INITIATING COMPLETE PIPELINE: PUNJABI ➔ HINDI TRANSLATION ➔ REPORT GENERATION

──────────────────────────────────────────────────
🚀 Processing Model: gemini-2.5-flash
──────────────────────────────────────────────────
   💾 Saved Translation File: Garden2006_turbo_gemini-2.5-flash_hindi.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   💾 Saved Analysis Report: Garden2006_turbo_gemini-2.5-flash_hindi_report.md


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

   ❌ Report Phase Failed for gemini-2.5-flash-lite: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

📁 Created separate archive dataset summary: Garden2006_turbo_pipeline_grand_total_summary.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All processes finished execution. All transcript texts and analytical markdown file structures have been compiled and sent to your downloads folder.


Comparison report



In [11]:
import os
from openai import OpenAI
from google.colab import userdata
from google.colab import files as colab_files

# 1. Initialize OpenRouter Client for the Meta-Judge
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
client_universal = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY")
)

# 2. File Parameters (Using your exact file naming configurations)
base_name = "Garden2006_turbo"
direct_report_file = f"{base_name}_gemini-2.5-flash_direct_report.md"       # Method 1
translation_report_file = f"{base_name}_gemini-2.5-flash_hindi_report.md"  # Method 2

print("🔍 Reading generated report files from session storage...")

# 3. Safely read Method 1 (Direct Punjabi ➔ Hindi Report)
if os.path.exists(direct_report_file):
    with open(direct_report_file, "r", encoding="utf-8") as f:
        direct_report_content = f.read().strip()
else:
    raise FileNotFoundError(f"Could not find Direct Report file: {direct_report_file}")

# 4. Safely read Method 2 (Punjabi ➔ Hindi Translation ➔ Hindi Report)
if os.path.exists(translation_report_file):
    with open(translation_report_file, "r", encoding="utf-8") as f:
        translation_report_content = f.read().strip()
else:
    raise FileNotFoundError(f"Could not find Translation-Based Report file: {translation_report_file}")

# 5. Construct the Meta-Evaluation Prompt
meta_judge_prompt = f"""
You are a Principal AI Evaluation Architect and Senior Linguist. Compare two different analytical reports generated from an original Punjabi transcript. Evaluate which methodology produces a higher-quality final report.

---
METHODOLOGY 1 REPORT (Directly from Punjabi to Hindi Report):
{direct_report_content}

---
METHODOLOGY 2 REPORT (Generated after reviewing the intermediate Hindi Translation file):
{translation_report_content}
---

Generate a definitive Methodology Comparison Report in Markdown. Your analysis must cover:
1. **Depth of Summary (सारांश की गहराई):** Which report extracted better thematic context and granularity?
2. **Linguistic Precision (भाषाई सटीकता):** Did analyzing the intermediate translation file help catch more cultural idioms and nuances, or did it introduce noise/dilution?
3. **Structure & Formatting (संरचना):** Which layout is more professional and readable?
4. **Final Verdict (अंतिम निर्णय):** Provide a clear, data-driven conclusion stating which approach is better for production pipelines and why.
"""

print("🦙 Passing both summaries to meta-llama/llama-3.3-70b-instruct for evaluation...")

# 6. Call the Meta-Judge
try:
    response = client_universal.chat.completions.create(
        model="meta-llama/llama-3.3-70b-instruct",
        messages=[{"role": "user", "content": meta_judge_prompt}],
        temperature=0.1
    )

    comparison_report = response.choices[0].message.content.strip()
    tokens_total = response.usage.total_tokens

    # 7. Save and Download Comparison Results
    output_filename = f"{base_name}_methodology_comparison_verdict.md"
    with open(output_filename, "w", encoding="utf-8") as out_f:
        out_f.write(comparison_report)

    print(f"\n💾 Saved Comparison Verdict: {output_filename}")
    colab_files.download(output_filename)

    # Print Token Metrics View
    print("\n" + "="*50)
    print(f"📊 META-JUDGE EXECUTION COMPLETE")
    print(f"Total Evaluation Tokens Spent: {tokens_total:,}")
    print("="*50)

except Exception as e:
    print(f"❌ Meta-Judge Execution Failed: {str(e)}")

🔍 Reading generated report files from session storage...
🦙 Passing both summaries to meta-llama/llama-3.3-70b-instruct for evaluation...

💾 Saved Comparison Verdict: Garden2006_turbo_methodology_comparison_verdict.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📊 META-JUDGE EXECUTION COMPLETE
Total Evaluation Tokens Spent: 10,406
